In [1]:
import json
import pandas as pd
from collections import Counter
import shap
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

In [2]:
import os
import sys
device = "cuda"

project_root = os.path.abspath("..")
sys.path.insert(0, project_root)

In [3]:
from data_gathering.utils.util import clean_text
from classification.utils_finetune import load_dataset, split_dataset

In [4]:
with open(project_root + "/params.json", 'r') as f:
    PARAMS = json.load(f)

In [ ]:
model_id = project_root + "/classification/regression_model/regression_model.pkl"
dataset = load_dataset()
print(f"Size of dataset: {len(dataset)}")
labels = list(dataset["label"].unique())

label2id, id2label, train_data, test_data, val_data = split_dataset(dataset, labels, train_size=PARAMS["train_split"], val_size=PARAMS["val_split"])

print(f'Train: {Counter(train_data["label"])}')
print(f'Test: {Counter(test_data["label"])}')
print(f'Val: {Counter(val_data["label"])}')

df = pd.read_csv("data/themes_from_thesis.csv", sep="\t")
primary_texts = [clean_text(x) for x in df["EXCERPT"]]

Cleaning text


100%|██████████| 3067/3067 [00:00<00:00, 177673.69it/s]

Size of dataset: 3067
{'HIGH': 0, 'MEDIUM': 1, 'LOW': 2}
{0: 'HIGH', 1: 'MEDIUM', 2: 'LOW'}
Train: Counter({1: 1156, 2: 883, 0: 414})
Test: Counter({1: 145, 2: 110, 0: 52})
Val: Counter({1: 144, 2: 111, 0: 52})


In [16]:
loaded_model_reg = pickle.load(open(model_id, 'rb'))

vectorizer = pickle.load(open(project_root + "/classification/regression_model/vectorizer.pkl", 'rb'))
X_train = vectorizer.transform(list(train_data["text"])).toarray()
explainer = shap.LinearExplainer(loaded_model_reg, X_train, silent=True)

In [17]:
print("Val data")
sample_texts_val = vectorizer.transform(list(val_data["text"])).toarray()
shap_values_val = explainer(sample_texts_val)
print("Primary texts")
sample_texts_primary = vectorizer.transform(list(primary_texts)).toarray()
shap_values_primary = explainer(sample_texts_primary)

with open(f"shap_values/shap_val_reg.pkl", "wb") as f:
        pickle.dump(shap_values_val, f)

with open(f"shap_values/shap_primary_reg.pkl", "wb") as f:
        pickle.dump(shap_values_primary, f)

Val data
Primary texts


In [18]:
with open(f"shap_values/shap_val_reg.pkl", "rb") as f:
        shap_values_val = pickle.load(f)

with open(f"shap_values/shap_primary_reg.pkl", "rb") as f:
        shap_values_primary = pickle.load(f)

shap_val = shap_values_val.values
data_val = sample_texts_val

shap_primary = shap_values_primary.values
data_primary = sample_texts_primary

In [40]:
shap_val[0, 1, 1]

np.float64(0.0)

In [32]:
len(feature_names)

2160

In [46]:
feature_names = vectorizer.get_feature_names_out()  # word → column mapping
from collections import defaultdict
def get_class_contribs(shap_values, feature_names, class_labels, n=20):
    class_contribs = [defaultdict(list) for _ in range(3)]
    class_contexts = [defaultdict(list) for _ in range(3)]

    for sv in shap_values:
        for i, f in enumerate(feature_names):
            for c in range(3):
                class_contribs[c][f].append(sv[i, c])
                class_contexts[c][f].append(f)      
     
    return class_contribs, class_contexts

In [47]:
val_contribs, val_contexts = get_class_contribs(shap_val, feature_names, loaded_model_reg.classes_, n=30)
primary_contribs, primary_contexts = get_class_contribs(shap_primary, feature_names, loaded_model_reg.classes_, n=30)

In [56]:
def print_attr(class_contribs, class_contexts, id2label, top_k=10, attr="positive"):

    agg = []
    output = f""
    for c in range(3):
        token_scores = {
            token: np.mean(vals)
            for token, vals in class_contribs[c].items()
        }
        agg.append(token_scores)
        

    for c in range(3):
        output += "\n"+"-"*40
        output = output + f"\n\nTop words for class {id2label[c]}:"
        if attr == "positive":
            sorted_tokens = sorted(
                [(token, score) for token, score in agg[c].items() if score > 0],
                key=lambda x: x[1],
                reverse=True
            )
        else:
            sorted_tokens = sorted(
                [(token, score) for token, score in agg[c].items() if score < 0],
                key=lambda x: x[1],
                reverse=False
            )

        for token, score in sorted_tokens[:top_k]:

            # choose one example context
            example_context = class_contexts[c][token][:3]

            output = output + f"\n\n{token}: {score:.4f}"
     
    
    return output

In [ ]:
print(print_attr(val_contribs, val_contexts, id2label, top_k=20, attr="positive"))
# can't really merge tokens since it depends on features from tfidf


----------------------------------------

Top words for class HIGH:

water: 0.0131

egypt: 0.0100

says: 0.0069

rome: 0.0062

alexandreia: 0.0055

terina: 0.0049

spain: 0.0048

massalia: 0.0044

nile: 0.0044

province: 0.0041

campania: 0.0040

district: 0.0035

mulberry: 0.0034

aenaria: 0.0033

mareotis: 0.0032

fish: 0.0032

trees: 0.0031

coast: 0.0031

mouths: 0.0031

parallel: 0.0031
----------------------------------------

Top words for class MEDIUM:

city: 0.0194

marsh: 0.0093

island: 0.0091

et: 0.0079

sailed: 0.0065

quinariae: 0.0043

waters: 0.0042

tiber: 0.0042

rhone: 0.0037

say: 0.0037

aspis: 0.0036

libya: 0.0033

bitter: 0.0033

water: 0.0033

aenaria: 0.0033

thou: 0.0032

strait: 0.0032

statue: 0.0031

nearly: 0.0030

syrtis: 0.0027
----------------------------------------

Top words for class LOW:

naples: 0.0144

rome: 0.0059

rhone: 0.0052

sirens: 0.0051

scylla: 0.0050

river: 0.0047

aspis: 0.0046

ostia: 0.0045

province: 0.0043

narbonne: 0.0038

c